# 04 — Preprocessing Final & Pembuatan Dataset Lengkap

Notebook ini menggabungkan kembali 3 split dataset (train/val/test), melakukan preprocessing
ulang, menambahkan label integer, dan menghasilkan **`dataset_final_lengkap.csv`** yang siap digunakan
untuk training model SVM dan BiLSTM.

---

## Alur Notebook
1. **Install & Import** — library yang dibutuhkan
2. **Load & Gabungkan** — baca `train_final.csv`, `val_final.csv`, `test_final.csv`
3. **Preprocessing Teks** — cleaning, normalisasi slang, stopword, stemming
4. **Label Encoding** — tambah `label_id` (integer) dan `label` (string)
5. **Simpan** — `dataset_final_lengkap.csv`
6. **Verifikasi** — cek shape, kolom, dan distribusi label

> **Output:** `dataset_final_lengkap.csv` (separator `;`) — dataset utama untuk training model

> **Catatan:** Kolom `teks_no_stop` digunakan sebagai input **BiLSTM**, sedangkan `teks_final` digunakan sebagai input **SVM**.

## 1. Install & Import

In [ ]:
!pip install sastrawi -q

import pandas as pd
import re
import string
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

# Inisialisasi Sastrawi
stemmer          = StemmerFactory().create_stemmer()
stopword_remover = StopWordRemoverFactory().create_stop_word_remover()

print('Import selesai')

## 2. Load & Gabungkan Dataset

Gabungkan kembali 3 split dari notebook `03_gabungan_dataset.ipynb` menjadi satu DataFrame
untuk diproses secara seragam.

In [ ]:
df_train = pd.read_csv('train_final.csv')
df_val   = pd.read_csv('val_final.csv')
df_test  = pd.read_csv('test_final.csv')

# Gabungkan menjadi satu DataFrame
df = pd.concat([df_train, df_val, df_test], ignore_index=True)

# Bersihkan kolom yang tidak diperlukan
df = df.drop(columns=['skor', 'source'], errors='ignore')
df = df.rename(columns={'emotion': 'label'})

print(f'Total data gabungan : {len(df)} baris')
print(f'Kolom               : {df.columns.tolist()}')
print()
print('Distribusi label:')
print(df['label'].value_counts())
df.head()

## 3. Preprocessing Teks

Fungsi `preprocess_data()` menghasilkan **dua versi teks**:
- `teks_no_stop` — setelah cleaning + normalisasi slang + stopword removal → input **BiLSTM**
- `teks_final` — hasil stemming dari `teks_no_stop` → input **SVM**

In [ ]:
# Kamus normalisasi slang sederhana
slang_dict = {
    'gak'  : 'tidak',
    'nggak': 'tidak',
    'yg'   : 'yang',
    'udah' : 'sudah',
    'bgt'  : 'banget',
    'kalo' : 'kalau',
    'bikin': 'membuat',
    'pas'  : 'saat',
}

def preprocess_data(text):
    """
    Preprocessing teks dalam 4 tahap:
    1. Cleaning dasar (lowercase, hapus URL, angka, tanda baca)
    2. Normalisasi slang
    3. Stopword removal → teks_no_stop (input BiLSTM)
    4. Stemming → teks_final (input SVM)
    """
    # 1. Cleaning
    text = str(text).lower()
    text = re.sub(r'https?://\S+|www\.\S+', '', text)  # hapus URL
    text = re.sub(r'[-+]?[0-9]+', '', text)              # hapus angka
    text = text.translate(str.maketrans('', '', string.punctuation))  # hapus tanda baca
    text = text.strip()

    # 2. Normalisasi slang
    words = text.split()
    words = [slang_dict.get(word, word) for word in words]
    text  = ' '.join(words)

    # 3. Stopword removal → input BiLSTM
    text_no_stop = stopword_remover.remove(text)

    # 4. Stemming → input SVM
    text_final = stemmer.stem(text_no_stop)

    return text_no_stop, text_final


print('Memproses teks... Mohon tunggu.')
df[['teks_no_stop', 'teks_final']] = df['normalize_text'].apply(
    lambda x: pd.Series(preprocess_data(x))
)
print('Selesai!')
df.head()

## 4. Label Encoding

Tambahkan kolom `label_id` (integer) untuk keperluan training model.

In [ ]:
# Mapping label string → integer
label2id = {
    'anger'       : 0,
    'anticipation': 1,
    'disgust'     : 2,
    'fear'        : 3,
    'joy'         : 4,
    'sadness'     : 5,
    'trust'       : 6,
}

df['label_id'] = df['label'].str.lower().map(label2id)

# Susun urutan kolom
df = df[['normalize_text', 'teks_no_stop', 'teks_final', 'label', 'label_id']]

print('Label encoding selesai')
print()
print('Mapping label:')
for k, v in label2id.items():
    print(f'  {v} → {k}')
print()
df.head()

## 5. Simpan Dataset Final

In [ ]:
# Simpan dengan separator ';' agar kolom teks tidak konflik dengan koma
df.to_csv('dataset_final_lengkap.csv', index=False, sep=';')

print('File tersimpan: dataset_final_lengkap.csv')

## 6. Verifikasi Dataset

In [ ]:
df_check = pd.read_csv('dataset_final_lengkap.csv', sep=';')

print(f'Shape  : {df_check.shape}')
print(f'Kolom  : {df_check.columns.tolist()}')
print()
print('Cek null:')
print(df_check.isnull().sum())
print()
print('Distribusi label:')
print(df_check['label'].value_counts())
print()
df_check.head(10)

## Ringkasan

| Item | Nilai |
|------|-------|
| Input | `train_final.csv`, `val_final.csv`, `test_final.csv` |
| Preprocessing | cleaning → slang normalization → stopword removal → stemming |
| Kolom teks BiLSTM | `teks_no_stop` |
| Kolom teks SVM | `teks_final` |
| Jumlah kelas | 7 emosi |
| Output | `dataset_final_lengkap.csv` (separator `;`) |
| Total baris | ~3.727 |